In [ ]:
from elucidated_diffusion.elucidated_diffusion import edm_ancestral_sampling_for_diffusion
from elucidated_diffusion.elucidated_diffusion import edm_ancestral_sampling_for_sr
from elucidated_diffusion.elucidated_diffusion import P_mean, P_std, sigma_data, edm_loss_weight
from elucidated_diffusion.ema_helper import EMAHelper
#from elucidated_diffusion.models.chatgpt_diffusion_unet import UNet128
from elucidated_diffusion.models.claude_skip_attention import SkipAttentionUNet
# from elucidated_diffusion.models.claude_cascaded_vit import FlexibleCascadedViT # old version
#from elucidated_diffusion.models.claude_cascaded_vit import MultiScaleSharedViT # old version in pixel space
#from elucidated_diffusion.models.claude_cascaded_vit import SemanticCascadeViT
#from elucidated_diffusion.models.claude_cascaded_vit import MultiScaleSemanticViT
from elucidated_diffusion.models.claude_cascaded_vit import CoarseToFineViT

from elucidated_diffusion.models.claude_diffusion_ViTEnhancedUNet64 import create_vit_enhanced_unet
from elucidated_diffusion.models.claude_diffusion_ViTGradualTransition import create_gradual_transition_unet
from elucidated_diffusion.models.claude_diffusion_ViTGradualTransition import create_gradual_transition_unet
from elucidated_diffusion.models.claude_SemanticCoordinatorViTUNet import create_semantic_coordinator
#from elucidated_diffusion.dataset_helpers import get_datasets
from elucidated_diffusion.checkpoint_helper import load_checkpoint, save_checkpoint, show_model_info

device='cpu'

In [ ]:
from elucidated_diffusion.models.claude_cortical_refiner import CorticalRefinerUNet
from torchview import draw_graph
import torch
import IPython.display as ipd
img_size=256

m = CorticalRefinerUNet(img_size=256, cnn_layers=4, vit_layers=1,
        vit_resolution=8, base_ch=128, vit_ch=256,t_scale=1000)


x,t = torch.rand(12, 3, img_size, img_size).to(device), torch.rand(12).to(device)

model_graph = draw_graph(m, input_data=(x,t), device='meta',
                        expand_nested=True,
                        roll=True,depth=2,graph_dir='UD')
visual_graph = model_graph.visual_graph
with visual_graph.subgraph() as s:
    s.attr(rank='same')
    s.node('0')
    s.node('1')
    #s.node('2')
model_graph.visual_graph
model_graph.visual_graph.render(filename='doc/tmp_graph', format='svg', cleanup=True)
model_graph.visual_graph.render(filename='doc/tmp_graph', format='pdf', cleanup=True)
svg_data = model_graph.visual_graph.pipe(format='svg')
ipd.display(ipd.SVG(svg_data))

print(show_model_info(m))

In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.utils as vutils
import torchvision.transforms as transforms
from elucidated_diffusion.image_helpers import pil_to_data_url, html_for_images
def generate_images(model_edm, num_steps=15, batch_size=20, img_shape = (3,64,64)):
    print("in genimgs ",img_shape)
    to_pil = transforms.ToPILImage()
    with torch.no_grad():
        samples = edm_ancestral_sampling_for_diffusion(model_edm, num_steps=num_steps, batch_size=batch_size, img_shape=img_shape).cpu()
        samples = (samples + 1) / 2  # scale from [-1,1] to [0,1]
        samples = samples.clamp(0, 1)
        pil_images = [to_pil(img) for img in samples]
    return pil_images

def sample_html(model, batch_size=12, img_shape=(3,64,64), num_steps=36, title="EDM Samples",min_height=128):
    print("in sample_haml ",img_shape)
    imgs = generate_images(model_edm=model, batch_size=batch_size, img_shape=img_shape, num_steps=16)
    h = html_for_images(imgs, title=title, min_height=min_height)
    return h

#cp = '../checkpoints/good/SkipAttentionUNet_2025-12-03_19-26-19_pokemon_efficient.pth'
#lm = load_checkpoint(m,None,cp)

m = CorticalRefinerUNet(img_size=256, cnn_layers=4, vit_layers=2,
        vit_resolution=8, base_ch=128, vit_ch=256,t_scale=1000)
m.eval()
cp = '../checkpoints/fantasy_256_CorticalRefinerUNet_4_2_big.pth_ema.pth'
lm = load_checkpoint(m,None,cp)
if show_that_it_works := False:
    h = sample_html(m,title=f"Samples from {cp}",min_height=128, batch_size=1, img_shape=(3,256,256))
    import IPython.display as ipd
    ipd.display(ipd.HTML(h))

In [ ]:
import torch
import numpy as np
from PIL import Image

def pil_to_tensor(pil_img):
    img = pil_img.resize((128, 128), Image.BILINEAR)
    arr = np.array(img, dtype=np.float32) / 255.0
    return torch.tensor(arr * 2.0 - 1.0).unsqueeze(0)  # Scale to [-1, 1]

def tensor_to_pil(arr):
    arr = (arr - arr.min()) / (arr.max() - arr.min())
    return transforms.ToPILImage()(arr)

def show_results(original,noised,denoised,title=None):
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    if title:
            fig.suptitle(main_title, fontsize=16)
    axes[0].imshow(tensor_to_pil(original), cmap="gray")
    axes[1].imshow(tensor_to_pil(noised), cmap="gray")
    axes[2].imshow(tensor_to_pil(denoised), cmap="gray")
    axes[0].set_title("original")
    axes[1].set_title("noised")
    axes[2].set_title("denoised")
    [ax.axis("off") for ax in axes]
    plt.show()
    
def add_noise(clean_tensor, noise_sigma):
    """
    Add noise at specified sigma level.
    Note: clean_tensor should be in range [-1, 1] (matching training)
    """
    noise = torch.randn_like(clean_tensor)
    noised = clean_tensor + noise_sigma * noise
    return noised


def add_noise(clean_tensor, noise_sigma, blur_sigma=0):
    from scipy.ndimage import gaussian_filter
    blurred = gaussian_filter(clean_tensor, sigma=blur_sigma)
    noise = np.random.randn(*clean_tensor.shape) * noise_sigma
    noised = (blurred + noise)
    #noised = 2*(noised - noised.min()) / (noised.max() - noised.min()) -1
    return torch.tensor(noised).float()

def denoise(noisy_tensor, model, num_steps=18, start_sigma=1.0):
    device = next(model.parameters()).device
    sigma_data = 0.5
    sigma_min = 0.002
    rho = 7.0
    
    # Ensure proper tensor handling
    if not isinstance(noisy_tensor, torch.Tensor):
        x = torch.from_numpy(noisy_tensor).float()
    else:
        x = noisy_tensor.clone()
    
    # Add batch dimension if needed
    if x.dim() == 3:
        x = x.unsqueeze(0)
    
    x = x.to(device)
    
    # Create schedule starting from start_sigma down to sigma_min
    step_indices = torch.arange(num_steps, dtype=torch.float64, device=device)
    t_steps = (start_sigma ** (1/rho) + step_indices / (num_steps - 1) * 
              (sigma_min ** (1/rho) - start_sigma ** (1/rho))) ** rho
    t_steps = torch.cat([t_steps, torch.zeros_like(t_steps[:1])])
    
    # NO SCALING HERE - the input should already be at the right noise level
    
    # Main denoising loop
    x_next = x
    for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])):
        x_cur = x_next  # Match the pattern from working code
        
        sigma = t_cur.float()
        c_skip = sigma_data ** 2 / (sigma ** 2 + sigma_data ** 2)
        c_out = sigma * sigma_data / (sigma ** 2 + sigma_data ** 2).sqrt()
        c_in = 1 / (sigma_data ** 2 + sigma ** 2).sqrt()
        c_noise = sigma.log() / 4
        
        # Model prediction
        F_x = model(c_in * x_cur, c_noise.expand(x_cur.shape[0]))
        denoised = c_skip * x_cur + c_out * F_x
        d_cur = (x_cur - denoised) / t_cur
        x_next = x_cur + (t_next - t_cur) * d_cur
        
        # Heun's 2nd order correction
        if i < num_steps - 1:
            sigma_next = t_next.float()
            c_skip_next = sigma_data ** 2 / (sigma_next ** 2 + sigma_data ** 2)
            c_out_next = sigma_next * sigma_data / (sigma_next ** 2 + sigma_data ** 2).sqrt()
            c_in_next = 1 / (sigma_data ** 2 + sigma_next ** 2).sqrt()
            c_noise_next = sigma_next.log() / 4
            
            F_x_next = model(c_in_next * x_next, c_noise_next.expand(x_next.shape[0]))
            denoised_next = c_skip_next * x_next + c_out_next * F_x_next
            d_prime = (x_next - denoised_next) / t_next
            x_next = x_cur + (t_next - t_cur) * (0.5 * d_cur + 0.5 * d_prime)
    
    return x_next.squeeze(0).cpu()

def try_noise_levels(model_edm, test_dataset,idx,noise_sigma=1,blur_sigma=0,resample_by=1,start_sigma=None):
    if start_sigma is None:
        start_sigma = noise_sigma
    original = test_dataset[idx][0]
    x = original
    import einx
    import torch
    x = einx.mean("c (h h_scale) (w w_scale) -> c h w", x, h_scale=resample_by, w_scale=resample_by)
    x = einx.rearrange("c h w -> c (h h_scale) (w w_scale)", x, h_scale=resample_by, w_scale=resample_by)
    noised = add_noise(x,noise_sigma,blur_sigma)
    denoised = denoise(noised,model_edm,num_steps=18,start_sigma=start_sigma)
    title = f"noise_sigma={noise_sigma}"
    show_results(original,noised,denoised)

from elucidated_diffusion.dataset_helpers import AugmentedHRLRDataset
from torch.utils.data import DataLoader

def get_dataset_and_loader(dataset_name,img_size):
    ds = AugmentedHRLRDataset(f"../data/256x256/{dataset_name}",img_size,img_size)
    batch_size = 1
    print(f"Using batch size of {batch_size}")
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True, num_workers=0)
    return ds, dl


In [ ]:
import random
import matplotlib.pyplot as plt
from torchvision import transforms

dataset_name='fantasy'
from elucidated_diffusion.models.claude_cortical_refiner import CorticalRefinerUNet
img_size=256
# m = CorticalRefinerUNet(img_size=256, cnn_layers=4, vit_layers=2,
#         vit_resolution=8, base_ch=128, vit_ch=256,t_scale=1000)
# cp = '../checkpoints/fantasy_256_CorticalRefinerUNet_4_2_big.pth_ema.pth'
# lm = load_checkpoint(m,None,cp)

test_dataset,test_loader = get_dataset_and_loader(dataset_name,img_size)
idx = random.randint(0,len(test_dataset))
# 
idx = 0
#try_noise_levels(m,test_dataset,idx,0.1)
# try_noise_levels(idx,1)
# try_noise_levels(idx,3)
# try_noise_levels(idx,5)

In [ ]:
!uv add ipywidgets

In [ ]:
%matplotlib widget
import ipywidgets as widgets
widgets.IntSlider()


In [ ]:
%matplotlib widget
import matplotlib.pyplot as plt
plt.plot([1,2,3])

In [ ]:
%matplotlib widget
#%matplotlib notebook
import IPython

from elucidated_diffusion.attention_visualizer import visualize_attention
import torch

orig,h,l = test_dataset[5]
orig.shape
import einx
# x = einx.id("c h w -> 1 c h w",orig)
# x.shape

m = CorticalRefinerUNet(img_size=256, cnn_layers=4, vit_layers=2,
        vit_resolution=8, base_ch=128, vit_ch=256,t_scale=1000)
m.eval()
cp = '../checkpoints/fantasy_256_CorticalRefinerUNet_4_2_big.pth_ema.pth'
lm = load_checkpoint(m,None,cp)

x = einx.id("c h w -> 1 c h w",orig) # or any [1, 3, H, W] tensor

noised = add_noise(x,1,0)

t = torch.tensor([0.5])

with torch.no_grad():
    out, attn_maps = m(noised, t, return_attn=True)

visualize_attention(attn_maps, background_image=x[0])

In [ ]:
1/0

In [ ]:
m.to('cuda')
idx=5
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,1,16,start_sigma=n*2)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('fantasy',img_size)
idx = random.randint(0,len(test_dataset))
idx = 3372
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,n,16, start_sigma = 2*n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('fantasy',img_size)
idx = random.randint(0,len(test_dataset))
idx = 3372
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,n,1, start_sigma = (n*n+n*n)**0.5)

In [ ]:
m.to('cuda')
with torch.inference_mode():
    for n in [4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
m.to('cuda')
idx=5
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
m.to('cuda')
idx=5
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,n,start_sigma=(n)**0.5)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('fantasy',img_size)
idx = random.randint(0,len(test_dataset))
#idx = 3372
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,n,8, start_sigma = n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('fantasy',img_size)
idx = random.randint(0,len(test_dataset))
idx = 3372
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,n,32, start_sigma = n + n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('fantasy',img_size)
idx = random.randint(0,len(test_dataset))
idx = 3372
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,n,32, start_sigma = n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('fantasy',img_size)
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,0,1,2, start_sigma = n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('fantasy',img_size)
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,n,32, start_sigma = (n*n+n*n)**0.5)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('fantasy',img_size)
idx = random.randint(0,len(test_dataset))
idx = 3372
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n,n,32, start_sigma = (n*n+n*n)**0.5)

In [ ]:
m.to('cuda')
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
m.to('cuda')
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('pokemon',img_size)
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('mnist_scaled',img_size)
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('mnist_scaled',img_size)
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('mnist_scaled',img_size)
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
m.to('cuda')
test_dataset,test_loader = get_dataset_and_loader('pokemon',img_size)
idx = random.randint(0,len(test_dataset))
print(idx)
with torch.inference_mode():
    for n in [1,2,4,8,16]:
        print(n)
        try_noise_levels(m,test_dataset,idx,n)

In [ ]:
from elucidated_diffusion.models.chatgpt_semantic_bottleneck import GlobalSemanticViT_EDM
m = GlobalSemanticViT_EDM(max_res=128)
print(show_model_info(m))

In [ ]:
from elucidated_diffusion.models.claude_cascaded_vit import CoarseToFineViT
m = CoarseToFineViT()
show_model_info(m)

In [ ]:
from elucidated_diffusion.models.claude_cascaded_vit import CoarseToFineViT
m = CoarseToFineViT()
show_model_info(m)


In [ ]:
from torchview import draw_graph
import torch
import IPython.display as ipd
img_size=128
model = CoarseToFineViT()
#model = UNet128(use_attention=True).to(device)
x,t = torch.rand(12, 3, img_size, img_size).to(device), torch.rand(12).to(device)


#print(x,t)
model_graph = draw_graph(model, input_data=(x,t), device='meta',
                        expand_nested=True,
                        roll=False,depth=3,graph_dir='LR')
visual_graph = model_graph.visual_graph
with visual_graph.subgraph() as s:
    s.attr(rank='same')
    s.node('0')
    s.node('1')
    #s.node('2')
model_graph.visual_graph
model_graph.visual_graph.render(filename='doc/unet128_graph', format='svg', cleanup=True)
model_graph.visual_graph.render(filename='doc/unet128_graph', format='pdf', cleanup=True)
svg_data = model_graph.visual_graph.pipe(format='svg')
ipd.display(ipd.SVG(svg_data))

In [ ]:
from elucidated_diffusion.models.claude_cascaded_vit import CoarseToFineViT
m = CoarseToFineViT()
show_model_info(m)


In [ ]:
from torchview import draw_graph
import torch
import IPython.display as ipd
img_size=128
model = CoarseToFineViT()
#model = UNet128(use_attention=True).to(device)
x,t = torch.rand(12, 3, img_size, img_size).to(device), torch.rand(12).to(device)


#print(x,t)
model_graph = draw_graph(model, input_data=(x,t), device='meta',
                        expand_nested=True,
                        roll=False,depth=3,graph_dir='LR')
visual_graph = model_graph.visual_graph
with visual_graph.subgraph() as s:
    s.attr(rank='same')
    s.node('0')
    s.node('1')
    #s.node('2')
model_graph.visual_graph
model_graph.visual_graph.render(filename='doc/unet128_graph', format='svg', cleanup=True)
model_graph.visual_graph.render(filename='doc/unet128_graph', format='pdf', cleanup=True)
svg_data = model_graph.visual_graph.pipe(format='svg')
ipd.display(ipd.SVG(svg_data))

In [ ]:
from elucidated_diffusion.models.tmp import MultiScaleSharedViT
m = MultiScaleSharedViT()
show_model_info(m)


In [ ]:
from elucidated_diffusion.models.claude_cascaded_vit import MultiScaleSemanticViT
m = MultiScaleSemanticViT(patch_size=2)
show_model_info(m)


In [ ]:
from elucidated_diffusion.models.claude_cascaded_vit import MultiScaleSemanticViT
m = MultiScaleSemanticViT()
show_model_info(m)


In [ ]:
from elucidated_diffusion.models.claude_cascaded_vit import MultiScaleSemanticViT
m = MultiScaleSemanticViT()
show_model_info(m)


In [ ]:

m = SemanticCascadeViT().to('cpu')
show_model_info(m)


In [ ]:

m = SemanticCascadeViT().to('cpu')
# Before weight sharing between layers
show_model_info(m)


In [ ]:

m = SemanticCascadeViT().to('cpu')
# Before weight sharing between layers
show_model_info(m)


In [ ]:
m = UNet128().to('cpu')
show_model_info(m)


In [ ]:

m = SkipAttentionUNet().to('cpu')
show_model_info(m)


In [ ]:
import torch
import matplotlib.pyplot as plt
import torchvision.utils as vutils
import torchvision.transforms as transforms
from elucidated_diffusion.image_helpers import pil_to_data_url, html_for_images
def generate_images(model_edm, num_steps=15, batch_size=20, img_shape = (3,64,64)):
    print("in genimgs ",img_shape)
    to_pil = transforms.ToPILImage()
    with torch.no_grad():
        samples = edm_ancestral_sampling_for_diffusion(model_edm, num_steps=num_steps, batch_size=batch_size, img_shape=img_shape).cpu()
        samples = (samples + 1) / 2  # scale from [-1,1] to [0,1]
        samples = samples.clamp(0, 1)
        pil_images = [to_pil(img) for img in samples]
    return pil_images

def sample_html(model, batch_size=12, img_shape=(3,64,64), num_steps=36, title="EDM Samples",min_height=128):
    print("in sample_haml ",img_shape)
    imgs = generate_images(model_edm=model, batch_size=batch_size, img_shape=img_shape, num_steps=16)
    h = html_for_images(imgs, title=title, min_height=min_height)
    return h

cp = '../checkpoints/good/SkipAttentionUNet_2025-12-03_19-26-19_pokemon_efficient.pth'
lm = load_checkpoint(m,None,cp)
h = sample_html(m,title=f"Samples from {cp}",min_height=128, batch_size=1, img_shape=(3,128*2,128*2))
import IPython.display as ipd
ipd.display(ipd.HTML(h))

In [ ]:
%%time
# Takes like 20GB of RAM and 15 minutes
h = sample_html(m,title=f"Samples from {cp}",min_height=128, batch_size=1, img_shape=(3,128*4,128*4))
import IPython.display as ipd
ipd.display(ipd.HTML(h))

In [ ]:
%%time
# Takes like 20GB of RAM and 15 minutes
h = sample_html(m,title=f"Samples from {cp}",min_height=128, batch_size=4, img_shape=(3,128,128))
import IPython.display as ipd
ipd.display(ipd.HTML(h))

In [ ]:
%%time
# Takes like 20GB of RAM and 15 minutes
h = sample_html(m,title=f"Samples from {cp}",min_height=128, batch_size=8, img_shape=(3,128,128))
import IPython.display as ipd
ipd.display(ipd.HTML(h))

In [ ]:
def generate_html_examples(cp="checkpoints/UNet128_2025-09-12_19-47-15_dragon.pth",batch_size=12, img_shape=(3,128,128)):
    with torch.no_grad():
        model_edm = UNet128(use_attention=True).to(device)
        optimizer_edm = None
        load_checkpoint(model_edm, optimizer_edm, cp)
        h = sample_html(model_edm,title=f"Samples from {cp}",min_height=1024, batch_size=batch_size, img_shape=img_shape)
        ipd.display(ipd.HTML(h))
        model_edm.to('cpu')


def demo_semantic_coordinator():
    img_size=128
    model_edm = create_semantic_coordinator(
        img_size=img_size,
        cnn_layers=3,        vit_layers=4,        cnn_resolution=16,        vit_resolution=16,
    ).to(device)
    cp = 'checkpoints/UnifiedSemanticCoordinatorUNet_2025-12-01_09-25-10_pokemon_efficient.pth'
    optimizer_edm = None
    load_checkpoint(model_edm, optimizer_edm, cp)
    h = sample_html(model_edm,  img_shape=(3,img_size,img_size), 
                                title=f"EDM of {model_edm.__class__.__name__}"
                                )
    ipd.display(ipd.HTML(h))
    model_edm.to('cpu')
demo_semantic_coordinator()



generate
        

In [ ]:
m =  create_semantic_coordinator(config='balanced', img_size=128)
show_model_info(m)

In [ ]:
m = create_gradual_transition_unet("balanced").to('cpu')
show_model_info(m)


In [ ]:
m =  create_vit_enhanced_unet("balanced")
show_model_info(m)

In [ ]:

#model_edm = SkipAttentionUNet().to(device) # Train one of these longer -- they're awesome at pokemon # 2025-12-02
#model_edm = MultiScaleSharedViT().to(device)
model_edm = SemanticCascadeViT().to(device)